# Laboratory 5: Quantum Measurements
## Superposition, State Collapse, Spin & Polarization on the Bloch Sphere

**Course:** Fundamental Concepts of Quantum Technologies  
**Credits:** 1 ECTS &nbsp;|&nbsp; **Mode:** Online Simulators + Analytical Companion

---

### About This Notebook

This lab uses **external visual simulators** for Activities 1–4 (see the HTML guide). This notebook is the
**analytical companion**: it lets you compute, verify, and visualize the same physics using Qiskit's
`Statevector` and simulation tools — reinforcing what you observe in the simulators with exact mathematics.

**Simulators used in the HTML lab:**
- [QuVis Bloch Sphere](https://www.st-andrews.ac.uk/physics/quvis/) — Activities 1–2
- [Quantum Flytrap](https://lab.quantumflytrap.com/) — Activity 3
- [PhET Stern-Gerlach](https://phet.colorado.edu/en/simulations/stern-gerlach) — Activity 4
- [IBM Quantum Composer](https://quantum.ibm.com/composer) — Activity 5

---

**Learning Objectives:**
1. Visualize qubit states on the Bloch sphere programmatically.
2. Compute measurement probabilities using the Born rule analytically.
3. Simulate sequential measurements to demonstrate non-commutativity.
4. Map photon polarization angles to Bloch sphere coordinates.
5. Analyze spin measurement probabilities in the X, Y, Z bases.

---
## Environment Setup

Run this cell first to install and import all required libraries.

In [ ]:
!pip install qiskit[visualization] qiskit-aer matplotlib scipy pylatexenc numpy --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector, DensityMatrix
from qiskit.visualization import (
    plot_histogram,
    plot_bloch_multivector,
    plot_state_qsphere,
)

print("\n  ENVIRONMENT SETUP COMPLETE — Lab 5: Quantum Measurements")

---
## Theoretical Background

### The Measurement Postulate

For a qubit in state $|\psi\rangle$ measured in basis $\{|m\rangle\}$, the **Born rule** gives:

$$P(m) = |\langle m|\psi\rangle|^2$$

After the measurement, the state **collapses** irreversibly:

$$|\psi\rangle \;\xrightarrow{\text{measure, outcome }m}\; |m\rangle$$

### Bloch Sphere Parametrization

Any single-qubit pure state:
$$|\psi\rangle = \cos\!\left(\frac{\theta}{2}\right)|0\rangle + e^{i\varphi}\sin\!\left(\frac{\theta}{2}\right)|1\rangle$$

| State | $\theta$ | $\varphi$ | Position |
|---|---|---|---|
| $\|0\rangle$ | $0$ | any | North Pole |
| $\|1\rangle$ | $\pi$ | any | South Pole |
| $\|{+}\rangle$ | $\pi/2$ | $0$ | $+X$ axis |
| $\|{-}\rangle$ | $\pi/2$ | $\pi$ | $-X$ axis |
| $\|i\rangle$ | $\pi/2$ | $\pi/2$ | $+Y$ axis |
| $\|{-i}\rangle$ | $\pi/2$ | $3\pi/2$ | $-Y$ axis |

### Non-Commutativity

$$[\hat{X}, \hat{Z}] = \hat{X}\hat{Z} - \hat{Z}\hat{X} = \begin{pmatrix}0&-2\\2&0\end{pmatrix} \neq 0$$

This means $\hat{X}$ and $\hat{Z}$ are **incompatible observables** — measuring one disturbs the other.

> **Little Endian Convention (Qiskit):** In a bitstring like `|10⟩`, the **rightmost** bit is qubit $q_0$
> and the **leftmost** bit is qubit $q_1$. This convention applies to all Qiskit measurement results.

---
## Exercise 1: Bloch Sphere Visualization

**Companion to Activity 1 (QuVis Bloch Sphere Simulator)**

Use `plot_bloch_multivector` to display user-defined states. Modify $\theta$ and $\varphi$ and observe
how the Bloch vector changes.

**Your Task:**
1. Run the cell with the default values to see the six cardinal states on the Bloch sphere.
2. Modify `THETA` and `PHI` in the STUDENT EXPERIMENT ZONE to explore arbitrary states.
3. Verify that states with the same $\theta$ but different $\varphi$ (e.g. $|{+}\rangle$ and $|{-}\rangle$)
   have the same Z-measurement probabilities.

In [ ]:
def bloch_state(theta, phi):
    """Return a Statevector for the Bloch sphere angles (theta, phi)."""
    return Statevector([
        np.cos(theta / 2),
        np.exp(1j * phi) * np.sin(theta / 2)
    ])

# Six cardinal states
cardinal_states = {
    r"|0⟩": bloch_state(0, 0),
    r"|1⟩": bloch_state(np.pi, 0),
    r"|+⟩": bloch_state(np.pi/2, 0),
    r"|−⟩": bloch_state(np.pi/2, np.pi),
    r"|i⟩": bloch_state(np.pi/2, np.pi/2),
    r"|−i⟩": bloch_state(np.pi/2, 3*np.pi/2),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 9), subplot_kw=dict(projection='3d'))
fig.suptitle("Six Cardinal States on the Bloch Sphere", fontsize=14, fontweight='bold')

for ax, (label, state) in zip(axes.flatten(), cardinal_states.items()):
    plot_bloch_multivector(state, ax=ax, title=label)

plt.tight_layout()
plt.show()

In [ ]:
# --- STUDENT EXPERIMENT ZONE ---
THETA = np.pi / 3   # polar angle in radians (0 = north pole, pi = south pole)
PHI   = np.pi / 4   # azimuthal angle in radians (0 to 2*pi)
# --------------------------------

custom_state = bloch_state(THETA, PHI)
print(f"State vector: {custom_state.data.round(4)}")
print(f"P(|0⟩) = {abs(custom_state.data[0])**2:.4f}")
print(f"P(|1⟩) = {abs(custom_state.data[1])**2:.4f}")

plot_bloch_multivector(custom_state)

---
## Exercise 2: Born Rule Calculator

**Companion to Activity 2 (Measurement in Different Bases)**

Given a state $|\psi\rangle$ and a measurement basis, compute $P(0)$ and $P(1)$ analytically
and verify with a Qiskit simulation.

**Your Task:**
1. Run the cell with the default state $|{+}\rangle$ to see the Born rule in action.
2. Try different states and bases in the STUDENT EXPERIMENT ZONE.
3. Verify that the simulated counts match the theoretical probabilities as shots → ∞.

In [ ]:
# Z-basis eigenstates
ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)

# X-basis eigenstates
ket_plus  = np.array([1, 1], dtype=complex) / np.sqrt(2)
ket_minus = np.array([1, -1], dtype=complex) / np.sqrt(2)

# Y-basis eigenstates
ket_i     = np.array([1,  1j], dtype=complex) / np.sqrt(2)
ket_neg_i = np.array([1, -1j], dtype=complex) / np.sqrt(2)

BASES = {
    'Z': (ket0, ket1, ['|0⟩', '|1⟩']),
    'X': (ket_plus, ket_minus, ['|+⟩', '|−⟩']),
    'Y': (ket_i, ket_neg_i, ['|i⟩', '|−i⟩']),
}

def born_rule(state_vec, basis='Z'):
    e0, e1, labels = BASES[basis]
    p0 = abs(np.dot(e0.conj(), state_vec))**2
    p1 = abs(np.dot(e1.conj(), state_vec))**2
    print(f"\n  State: {np.round(state_vec, 4)}")
    print(f"  Basis: {basis}")
    print(f"  P({labels[0]}) = {p0:.4f}  ({p0*100:.1f}%)")
    print(f"  P({labels[1]}) = {p1:.4f}  ({p1*100:.1f}%)")
    print(f"  Sum = {p0+p1:.6f}  (should be 1.0)")
    return p0, p1

# --- STUDENT EXPERIMENT ZONE ---
STATE = ket_plus    # Try: ket0, ket1, ket_plus, ket_minus, ket_i,
                    #      or bloch_state(np.pi/3, np.pi/4).data
MEASUREMENT_BASIS = 'X'  # Try: 'Z', 'X', 'Y'
NUM_SHOTS = 2000
# --------------------------------

p0_theory, p1_theory = born_rule(STATE, basis=MEASUREMENT_BASIS)

In [ ]:
# Verify with Qiskit simulation in the chosen basis
# For X-basis: apply H before measurement. For Y-basis: apply Sdg then H.

sv = Statevector(STATE)
qc = QuantumCircuit(1, 1)
qc.initialize(STATE, 0)

if MEASUREMENT_BASIS == 'X':
    qc.h(0)
elif MEASUREMENT_BASIS == 'Y':
    qc.sdg(0)
    qc.h(0)

qc.measure(0, 0)
display(qc.draw('mpl', style='iqp'))

counts = AerSimulator().run(qc, shots=NUM_SHOTS).result().get_counts()
print(f"\nSimulated ({NUM_SHOTS} shots): {counts}")
print(f"Simulated P(0) ≈ {counts.get('0', 0)/NUM_SHOTS:.4f}  (theory: {p0_theory:.4f})")
print(f"Simulated P(1) ≈ {counts.get('1', 0)/NUM_SHOTS:.4f}  (theory: {p1_theory:.4f})")

plot_histogram(counts, title=f"Measurement in {MEASUREMENT_BASIS}-basis ({NUM_SHOTS} shots)")

---
## Exercise 3: Sequential Measurement Simulation

**Companion to Activity 5 (Sequential Measurements & Non-Commutativity)**

Programmatically simulate the Z → X → Z measurement sequence over N shots
and compare it to Z → Z → Z. This demonstrates that non-commuting measurements
introduce irreducible randomness.

**Your Task:**
1. Run with `N_SHOTS = 1000` and compare the two histograms.
2. Increase `N_SHOTS` to see the statistics converge.
3. Note the Little Endian convention in the bitstrings.

In [ ]:
# --- STUDENT EXPERIMENT ZONE ---
N_SHOTS = 1000
# --------------------------------

# Sequence 1: Z → X → Z
# Measure Z (qc records c[0]), then measure X (apply H, measure → c[1]),
# then measure Z again (apply H to undo basis rotation → c[2])
# We use 3-qubit reset trick: mid-circuit measurement + fresh qubit prep
# Simplified: simulate as two separate steps by resetting state post-measurement.

def simulate_z_x_z(n_shots):
    """Simulate Z→X→Z by tracking the post-measurement state each time."""
    final_outcomes = []
    for _ in range(n_shots):
        state = np.array([1, 0], dtype=complex)  # start |0⟩

        # Measure Z
        p0 = abs(state[0])**2
        outcome_z1 = 0 if np.random.random() < p0 else 1
        state = np.array([1, 0]) if outcome_z1 == 0 else np.array([0, 1])

        # Measure X: compute projections onto |+⟩ and |−⟩
        p_plus = abs((state[0] + state[1]) / np.sqrt(2))**2
        outcome_x = '+' if np.random.random() < p_plus else '-'
        state = np.array([1, 1]) / np.sqrt(2) if outcome_x == '+' else np.array([1, -1]) / np.sqrt(2)

        # Measure Z again
        p0_final = abs(state[0])**2
        outcome_z2 = 0 if np.random.random() < p0_final else 1
        final_outcomes.append(outcome_z2)

    count_0 = sum(1 for o in final_outcomes if o == 0)
    count_1 = len(final_outcomes) - count_0
    return {'0': count_0, '1': count_1}


def simulate_z_z_z(n_shots):
    """Simulate Z→Z→Z: always stays |0⟩ after first Z measurement."""
    return {'0': n_shots, '1': 0}


counts_zxz = simulate_z_x_z(N_SHOTS)
counts_zzz = simulate_z_z_z(N_SHOTS)

print("Z→X→Z final Z measurement:", counts_zxz)
print("Z→Z→Z final Z measurement:", counts_zzz)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(counts_zxz.keys(), counts_zxz.values(), color=['steelblue', 'salmon'])
ax1.set_title(f'Z→X→Z (final Z), n={N_SHOTS}')
ax1.set_ylabel('Counts')
ax1.set_ylim(0, N_SHOTS)
for k, v in counts_zxz.items():
    ax1.text(k, v + 10, f'{v/N_SHOTS:.2f}', ha='center', fontweight='bold')

ax2.bar(counts_zzz.keys(), counts_zzz.values(), color=['steelblue', 'salmon'])
ax2.set_title(f'Z→Z→Z (final Z), n={N_SHOTS}')
ax2.set_ylabel('Counts')
ax2.set_ylim(0, N_SHOTS)
for k, v in counts_zzz.items():
    ax2.text(k, v + 10, f'{v/N_SHOTS:.2f}', ha='center', fontweight='bold')

plt.suptitle('Sequential Measurement: Non-Commutativity in Action', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nZ→X→Z: The X measurement collapses |0⟩ to |+⟩ or |−⟩, then Z gives random result.")
print("Z→Z→Z: Repeated Z measurements on a Z-eigenstate give the same result every time.")

### Verify Z→X→Z with Qiskit Circuits

The circuit below uses `reset` + mid-circuit operations to implement Z→X→Z in Qiskit.
Note the **Little Endian** bitstring ordering in the output counts.

In [ ]:
# Z-basis measurement, then X-basis (via H), then Z-basis again
# Using a 3-classical-bit register to record each measurement step
qc_seq = QuantumCircuit(1, 3)

# Start from |0⟩ — this is the initial Z measurement result
qc_seq.measure(0, 0)       # First Z measurement → c[0]

# X-basis measurement: rotate to X-basis, measure, rotate back
qc_seq.h(0)                # Rotate to X-basis
qc_seq.measure(0, 1)       # Measure → c[1]  (0='|+⟩', 1='|−⟩')
qc_seq.h(0)                # Rotate back: now in Z-eigenstate from X result

# Second Z measurement
qc_seq.measure(0, 2)       # Final Z measurement → c[2]

display(qc_seq.draw('mpl', style='iqp'))

# NOTE (Little Endian): The bitstring result reads c[2]c[1]c[0] right-to-left
counts_seq = AerSimulator().run(qc_seq, shots=N_SHOTS).result().get_counts()
print(f"\nBitstring format: c[2] c[1] c[0]  (Little Endian — rightmost bit = first measurement)")
print(f"Counts: {counts_seq}")

# Extract only the final Z result (leftmost bit in each bitstring)
final_z = {'0': 0, '1': 0}
for bitstring, count in counts_seq.items():
    final_bit = bitstring[0]  # leftmost = c[2] = final Z measurement
    final_z[final_bit] += count

print(f"\nFinal Z outcome only: {final_z}")
print(f"P(0) ≈ {final_z['0']/N_SHOTS:.3f}  (expected: 0.5)")
print(f"P(1) ≈ {final_z['1']/N_SHOTS:.3f}  (expected: 0.5)")

---
## Exercise 4: Polarization Mapping

**Companion to Activity 3 (Photon Polarization)**

Map photon polarization angles to Bloch sphere coordinates and visualize the equatorial plane.
A linearly polarized photon at angle $\alpha$ maps to the Bloch sphere state
with $\theta = \pi/2$ and $\varphi = 2\alpha$ (on the equator).

**Your Task:**
1. Run the visualization to see how polarization angles map to the Bloch equator.
2. Compute the Born rule transmission probability (Malus's Law) for different polarizer angles.
3. Modify `POLARIZER_ANGLE_DEG` to see the transmission change.

In [ ]:
def polarization_to_bloch(alpha_deg):
    """Map polarization angle alpha (degrees) to Bloch sphere (theta=pi/2, phi=2*alpha)."""
    alpha_rad = np.deg2rad(alpha_deg)
    phi_bloch = 2 * alpha_rad
    return bloch_state(np.pi/2, phi_bloch)

def malus_probability(photon_angle_deg, polarizer_angle_deg):
    """Born rule / Malus's Law: P(pass) = cos^2(polarizer - photon)."""
    delta = np.deg2rad(polarizer_angle_deg - photon_angle_deg)
    return np.cos(delta)**2

# Common polarization angles
angles_deg = [0, 45, 90, 135]
labels = ['H (0°)', 'Diagonal (45°)', 'V (90°)', 'Anti-diag (135°)']

states = [polarization_to_bloch(a) for a in angles_deg]

fig, axes = plt.subplots(1, 4, figsize=(16, 5), subplot_kw=dict(projection='3d'))
fig.suptitle("Photon Polarization → Bloch Sphere (Equatorial States)", fontsize=13, fontweight='bold')
for ax, state, label in zip(axes, states, labels):
    plot_bloch_multivector(state, ax=ax, title=label)
plt.tight_layout()
plt.show()

# Malus's Law table
print("\nMalus's Law / Born Rule — Transmission Probabilities")
print(f"{'Photon':>15} {'Polarizer':>15} {'P(pass)':>10} {'% pass':>8}")
print("-" * 52)
test_cases = [
    (0, 0), (0, 45), (0, 90),
    (45, 45), (45, 90), (45, 30),
]
for ph, pol in test_cases:
    p = malus_probability(ph, pol)
    print(f"  {ph:>3}° (photon)  →  {pol:>3}° (polarizer)  →  {p:.4f}  ({p*100:.1f}%)")

In [ ]:
# --- STUDENT EXPERIMENT ZONE ---
PHOTON_ANGLE_DEG    = 45    # photon polarization angle (degrees)
POLARIZER_ANGLE_DEG = 30    # polarizer orientation angle (degrees)
# --------------------------------

p_pass = malus_probability(PHOTON_ANGLE_DEG, POLARIZER_ANGLE_DEG)
photon_state = polarization_to_bloch(PHOTON_ANGLE_DEG)

print(f"Photon polarization: {PHOTON_ANGLE_DEG}°")
print(f"Polarizer angle:     {POLARIZER_ANGLE_DEG}°")
print(f"P(pass) = cos²({POLARIZER_ANGLE_DEG - PHOTON_ANGLE_DEG}°) = {p_pass:.4f}  ({p_pass*100:.1f}%)")
print(f"State vector: {photon_state.data.round(4)}")

plot_bloch_multivector(photon_state)

---
## Exercise 5: Spin State Analysis

**Companion to Activity 4 (Electron Spin & Stern-Gerlach)**

Define spin-up/spin-down states in the X, Y, Z axes. Compute and visualize measurement
probabilities for each axis combination.

**Your Task:**
1. Run the full analysis to see the probability table for all spin states × measurement axes.
2. Modify `SPIN_STATE` and `MEASURE_AXIS` in the STUDENT EXPERIMENT ZONE.
3. Verify that spin-up in Z gives 50/50 when measured in X — matching the Stern-Gerlach result.

In [ ]:
# Spin eigenstates along each axis
spin_states = {
    'Z_up':   np.array([1, 0], dtype=complex),
    'Z_down': np.array([0, 1], dtype=complex),
    'X_up':   np.array([1,  1], dtype=complex) / np.sqrt(2),
    'X_down': np.array([1, -1], dtype=complex) / np.sqrt(2),
    'Y_up':   np.array([1,  1j], dtype=complex) / np.sqrt(2),
    'Y_down': np.array([1, -1j], dtype=complex) / np.sqrt(2),
}

axes = {
    'Z': (np.array([1,0],dtype=complex), np.array([0,1],dtype=complex)),
    'X': (np.array([1,1],dtype=complex)/np.sqrt(2), np.array([1,-1],dtype=complex)/np.sqrt(2)),
    'Y': (np.array([1,1j],dtype=complex)/np.sqrt(2), np.array([1,-1j],dtype=complex)/np.sqrt(2)),
}

print("Spin State Measurement Probability Table")
print(f"{'State':>10}", end='')
for axis in axes:
    print(f"  P({axis}↑)  P({axis}↓)", end='')
print()
print("-" * 62)

for state_name, state_vec in spin_states.items():
    print(f"{state_name:>10}", end='')
    for axis, (e_up, e_down) in axes.items():
        p_up   = abs(np.dot(e_up.conj(), state_vec))**2
        p_down = abs(np.dot(e_down.conj(), state_vec))**2
        print(f"   {p_up:.3f}    {p_down:.3f}", end='')
    print()

print()
print("Key insight: Z_up measured in X → 50%/50% (Stern-Gerlach result from Activity 4)")

In [ ]:
# --- STUDENT EXPERIMENT ZONE ---
SPIN_STATE   = 'Z_up'   # Options: 'Z_up','Z_down','X_up','X_down','Y_up','Y_down'
MEASURE_AXIS = 'X'      # Options: 'Z', 'X', 'Y'
N_SHOTS      = 1000
# --------------------------------

state_vec = spin_states[SPIN_STATE]
e_up, e_down = axes[MEASURE_AXIS]

p_up   = abs(np.dot(e_up.conj(), state_vec))**2
p_down = abs(np.dot(e_down.conj(), state_vec))**2

print(f"Spin state:   |{SPIN_STATE}⟩ = {np.round(state_vec, 4)}")
print(f"Measure axis: {MEASURE_AXIS}")
print(f"P(↑_{MEASURE_AXIS}) = {p_up:.4f}  ({p_up*100:.1f}%)")
print(f"P(↓_{MEASURE_AXIS}) = {p_down:.4f}  ({p_down*100:.1f}%)")

# Visualize on Bloch sphere
fig, axes_plot = plt.subplots(1, 2, figsize=(10, 5), subplot_kw=dict(projection='3d'))
plot_bloch_multivector(Statevector(state_vec), ax=axes_plot[0], title=f"Spin state: {SPIN_STATE}")

# Post-measurement states weighted by probability
post_up   = Statevector(e_up)
post_down = Statevector(e_down)
label = f"After {MEASURE_AXIS}-meas: {p_up:.2f}×↑ + {p_down:.2f}×↓"
plot_bloch_multivector(post_up, ax=axes_plot[1], title=f"Post-meas ↑ state ({MEASURE_AXIS}-axis)")

plt.tight_layout()
plt.show()

# Qiskit simulation
qc_spin = QuantumCircuit(1, 1)
qc_spin.initialize(state_vec, 0)
if MEASURE_AXIS == 'X':
    qc_spin.h(0)
elif MEASURE_AXIS == 'Y':
    qc_spin.sdg(0)
    qc_spin.h(0)
qc_spin.measure(0, 0)

counts_spin = AerSimulator().run(qc_spin, shots=N_SHOTS).result().get_counts()
print(f"\nQiskit simulation ({N_SHOTS} shots): {counts_spin}")
print(f"Simulated P(↑) ≈ {counts_spin.get('0', 0)/N_SHOTS:.3f}  (theory: {p_up:.3f})")

plot_histogram(counts_spin, title=f"{SPIN_STATE} measured in {MEASURE_AXIS}-basis")

---
## Summary

This notebook provided the analytical companion to the Lab 5 simulator activities.

| Exercise | Concept demonstrated | Key result |
|---|---|---|
| 1 | Bloch sphere visualization | Cardinal states at poles and equator |
| 2 | Born rule calculator | $P(m) = |\langle m|\psi\rangle|^2$ verified by simulation |
| 3 | Sequential measurements | Z→X→Z gives 50/50 final Z; Z→Z→Z gives deterministic result |
| 4 | Polarization mapping | Malus's Law is the Born rule for photon polarization |
| 5 | Spin state analysis | Incompatible axes give 50/50; compatible axes give deterministic result |

**Little Endian reminder:** In Qiskit, bitstring `'10'` means $q_1 = 1$, $q_0 = 0$.
Rightmost bit = lowest qubit index.

---
*Quantum Computing Foundation — Lab 5: Quantum Measurements*